# World pop

Written by Ines, May 19 2026.

In [ ]:
#pip install rasterio numpy pandas geopandas xarray rio-cogeo stackstac pyproj


In [ ]:
import requests, pathlib

ISO3 = "AGO"
YEAR = 2020
url = f"https://data.worldpop.org/GIS/Population_Density/Global_2000_2020_1km_UNadj/{YEAR}/{ISO3}/{ISO3.lower()}_pd_{YEAR}_1km_UNadj.tif"
r = requests.get(url, stream=True)
pathlib.Path(f"{ISO3}_pop.tif").write_bytes(r.content)

In [ ]:
import rasterio, numpy as np, pandas as pd
from rasterio.transform import xy

with rasterio.open("AGO_pop.tif") as src:
    data = src.read(1).astype("float64")
    nodata = src.nodata
    transform = src.transform
    
data[data == nodata] = np.nan

rows, cols = np.where(~np.isnan(data))
lons, lats = xy(transform, rows, cols)

df = pd.DataFrame({
    "lat": lats,
    "lon": lons,
    "pop_density": data[rows, cols]
})

# main loop (download + extraction)

In [ ]:
import requests, pathlib, time
import rasterio, numpy as np, pandas as pd
from rasterio.transform import xy

YEAR = 2020

iso3_codes = [
    "AFG","ALB","DZA","AND","AGO","ATG","ARG","ARM","AUS","AUT","AZE","BHS","BHR","BGD","BRB",
    "BLR","BEL","BLZ","BEN","BTN","BOL","BIH","BWA","BRA","BRN","BGR","BFA","BDI","CPV","KHM",
    "CMR","CAN","CAF","TCD","CHL","CHN","COL","COM","COD","COG","CRI","CIV","HRV","CUB","CYP",
    "CZE","DNK","DJI","DOM","ECU","EGY","SLV","GNQ","ERI","EST","SWZ","ETH","FJI","FIN","FRA",
    "GAB","GMB","GEO","DEU","GHA","GRC","GTM","GIN","GNB","GUY","HTI","HND","HUN","ISL","IND",
    "IDN","IRN","IRQ","IRL","ISR","ITA","JAM","JPN","JOR","KAZ","KEN","PRK","KOR","KWT","KGZ",
    "LAO","LVA","LBN","LSO","LBR","LBY","LTU","LUX","MDG","MWI","MYS","MDV","MLI","MLT","MRT",
    "MUS","MEX","MDA","MNG","MNE","MAR","MOZ","MMR","NAM","NPL","NLD","NZL","NIC","NER","NGA",
    "MKD","NOR","OMN","PAK","PAN","PNG","PRY","PER","PHL","POL","PRT","QAT","ROU","RUS","RWA",
    "WSM","STP","SAU","SEN","SRB","SLE","SGP","SVK","SVN","SLB","SOM","ZAF","SSD","ESP","LKA",
    "SDN","SUR","SWE","CHE","SYR","TWN","TJK","TZA","THA","TLS","TGO","TTO","TUN","TUR","TKM",
    "UGA","UKR","ARE","GBR","USA","URY","UZB","VUT","VEN","VNM","YEM","ZMB","ZWE"
]

BASE_URL = "https://data.worldpop.org/GIS/Population_Density/Global_2000_2020_1km_UNadj"
output_dir = pathlib.Path("worldpop_2020")
output_dir.mkdir(exist_ok=True)

failed_download = []
failed_extract = []
all_dfs = []

def download_tif(iso3, year, output_dir):
    """Download tif file, return path or None on failure."""
    out_file = output_dir / f"{iso3}_pd_{year}.tif"
    if out_file.exists():
        print(f"[SKIP] {iso3} already exists")
        return out_file

    url = f"{BASE_URL}/{year}/{iso3}/{iso3.lower()}_pd_{year}_1km_UNadj.tif"
    try:
        r = requests.get(url, stream=True, timeout=60)
        if r.status_code == 200:
            out_file.write_bytes(r.content)
            print(f"[OK]   {iso3} → {out_file} ({len(r.content)/1e6:.1f} MB)")
            return out_file
        else:
            print(f"[FAIL] {iso3} — HTTP {r.status_code}")
            failed_download.append((iso3, r.status_code))
            return None
    except requests.exceptions.RequestException as e:
        print(f"[ERR]  {iso3} — {e}")
        failed_download.append((iso3, str(e)))
        return None

def extract_population(tif_path, iso3):
    """Extract lat, lon, pop_density from tif, return DataFrame or None on failure."""
    try:
        with rasterio.open(tif_path) as src:
            data = src.read(1).astype("float64")
            nodata = src.nodata
            transform = src.transform

        data[data == nodata] = np.nan
        rows, cols = np.where(~np.isnan(data))
        lons, lats = xy(transform, rows, cols)

        df = pd.DataFrame({
            "iso3": iso3,
            "lat": lats,
            "lon": lons,
            "pop_density": data[rows, cols]
        })
        print(f"[EXTRACTED] {iso3} — {len(df):,} valid pixels")
        return df

    except Exception as e:
        print(f"[EXTRACT ERR] {iso3} — {e}")
        failed_extract.append((iso3, str(e)))
        return None

# ── Main loop ──────────────────────────────────────────────────────────────────
for iso3 in iso3_codes:

    # 1. Download
    tif_path = download_tif(iso3, YEAR, output_dir)
    if tif_path is None:
        continue

    # 2. Extract
    df = extract_population(tif_path, iso3)
    if df is not None:
        all_dfs.append(df)

    time.sleep(0.5)

# ── Combine & save ─────────────────────────────────────────────────────────────
if all_dfs:
    combined = pd.concat(all_dfs, ignore_index=True)
    out_csv = pathlib.Path("worldpop_2020_all_countries.csv")
    combined.to_csv(out_csv, index=False)
    print(f"\nSaved {len(combined):,} rows → {out_csv}")
    print(combined.head())

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\nCountries processed : {len(all_dfs)}/{len(iso3_codes)}")
if failed_download:
    print(f"Download failures  : {failed_download}")
if failed_extract:
    print(f"Extraction failures : {failed_extract}")

In [ ]:
print(combined.head())